In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [5]:
# ==============================
# 1. Install (run once if needed)
# ==============================
!pip install transformers peft accelerate bitsandbytes

# ==============================
# 2. Imports
# ==============================
import torch
import re
import json
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# ==============================
# 3. Config
# ==============================
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER    = "Ravi2003/medscript-qwen2.5-3b-qlora"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ==============================
# 4. System Prompt (IMPROVED)
# ==============================
SYSTEM_PROMPT = """You are a clinical documentation assistant. Given an unstructured doctor's note, return ONLY a valid JSON object with exactly these 4 fields:

{
  "subjective": "What the patient reports — symptoms, history, complaints.",
  "objective": "Measurable clinical findings — vitals, labs, ECG, exam.",
  "assessment": "Diagnosis or clinical impression based on S and O.",
  "plan": "Treatment, medications, referrals, and follow-up steps."
}

Rules:
- Do NOT repeat information across sections.
- Do NOT include labels like "S:", "O:", "A:", "P:" inside section values.
- Do NOT add any text outside the JSON object.
- Keep each section concise, complete, and clinically accurate.
- If a field has no relevant information, use "Not documented."
- Ensure the output is strictly valid JSON with correct commas and syntax.
"""

# ==============================
# 5. Load tokenizer
# ==============================
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# ==============================
# 6. Load BASE model
# ==============================
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)
base_model.eval()

# ==============================
# 7. Load FINE-TUNED model
# ==============================
ft_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)
ft_model = PeftModel.from_pretrained(ft_base, ADAPTER)
ft_model.eval()

# ==============================
# 8. Input (same for both)
# ==============================
input_text = """
Patient is a 45 year old male presenting with chest pain for the past 2 hours.
Pain is pressure-like and radiates to the left arm. Patient has history of
hypertension and is on lisinopril. Blood pressure 150/90, heart rate 98,
oxygen saturation 97%. ECG shows ST elevation in leads II, III and aVF.
"""

# ==============================
# 9. Safe JSON extractor
# ==============================
def safe_json_extract(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except:
            return None
    return None

# ==============================
# 10. Shared generation function (FIXED)
# ==============================
def generate_with_model(model, note):
    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{note}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,          # 🔥 deterministic
            temperature=0.0,          # 🔥 no randomness
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    return decoded

# ==============================
# 11. Run BOTH
# ==============================
base_output = generate_with_model(base_model, input_text)
ft_output   = generate_with_model(ft_model, input_text)

# ==============================
# 12. Try parsing (for validation)
# ==============================
base_json = safe_json_extract(base_output)
ft_json   = safe_json_extract(ft_output)

# ==============================
# 13. Print results
# ==============================
print("\n" + "="*60)
print("BASE MODEL OUTPUT")
print("="*60)
print(base_output)

print("\nParsed JSON:", base_json)

print("\n" + "="*60)
print("FINE-TUNED MODEL OUTPUT")
print("="*60)
print(ft_output)

print("\nParsed JSON:", ft_json)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



BASE MODEL OUTPUT
{
  "subjective": "Chest pain for 2 hours described as pressure-like, radiating to left arm; history of hypertension on lisinopril; blood pressure 150/90, heart rate 98, oxygen saturation 97%; ECG shows ST elevation in leads II, III and aVF.",
  "objective": "ECG shows ST elevation in leads II, III and aVF; BP 150/90, HR 98, SpO2 97%.",
  "assessment": "Acute Coronary Syndrome (STEMI).",
  "plan": "Immediate PCI; aspirin, heparin, and beta-blocker administration; further cardiac monitoring; anticoagulation therapy;"
}

Parsed JSON: {'subjective': 'Chest pain for 2 hours described as pressure-like, radiating to left arm; history of hypertension on lisinopril; blood pressure 150/90, heart rate 98, oxygen saturation 97%; ECG shows ST elevation in leads II, III and aVF.', 'objective': 'ECG shows ST elevation in leads II, III and aVF; BP 150/90, HR 98, SpO2 97%.', 'assessment': 'Acute Coronary Syndrome (STEMI).', 'plan': 'Immediate PCI; aspirin, heparin, and beta-blocker 